<a href="https://colab.research.google.com/github/RobertoGarrahan/proa-mvp2/blob/main/MVP2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MVP: Engenharia de Software para Sistemas Inteligentes
## Projeto P.R.O.A. - Plataforma de Risco e Operação de Ativos

**Contexto do Problema:**
No mercado offshore, a confiabilidade dos ativos marítimos é um fator crítico para a segurança da tripulação e para a eficiência operacional. Falhas não planejadas em equipamentos de missão crítica (como propulsores, motores principais ou guinchos) resultam em alto *downtime* (tempo de inatividade da embarcação) e custos elevados de manutenção corretiva.

**Objetivo do Projeto:**
Este notebook atua como o relatório de desenvolvimento do "cérebro" analítico da plataforma P.R.O.A. O objetivo é construir, treinar e otimizar um modelo de *Machine Learning* para classificação binária capaz de atuar na **Manutenção Preditiva**. Através da leitura de dados de telemetria (sensores de temperatura, rotação e esforço), o sistema deve prever o risco de falha iminente de um ativo. O modelo final otimizado será exportado para ser consumido por uma aplicação web Full Stack.

**Sobre os Dados:**
Utilizamos uma base de dados de manutenções preditivas adaptável à realidade de equipamentos industriais pesados (*AI4I 2020 Predictive Maintenance Dataset*, do UCI Machine Learning Repository). O dataset possui 10.000 instâncias e monitora variáveis como:
* `Air temperature [K]`: Temperatura ambiente.
* `Process temperature [K]`: Temperatura de operação do equipamento.
* `Rotational speed [rpm]`: Rotação do motor/eixo.
* `Torque [Nm]`: Esforço mecânico.
* `Tool wear [min]`: Tempo de uso acumulado (desgaste).
* **Alvo (Target):** `Machine failure` (0 = Equipamento Normal, 1 = Falha).

**Etapas Contempladas neste Relatório:**
1. Carga do dataset via URL.
2. Separação de dados entre Treino e Teste (*Holdout*).
3. Tratamento de *Data Leakage* e transformação através de Pipelines (`StandardScaler`).
4. Treinamento de *Baseline* com algoritmos clássicos (KNN, Árvore de Decisão, Naive Bayes e SVM).
5. Otimização de Hiperparâmetros utilizando Validação Cruzada (`GridSearchCV`).
6. Análise de Métricas e Exportação do Modelo (`.pkl`).

In [7]:
import pandas as pd

# URL do dataset (conforme exigido: carga direta via URL)
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00601/ai4i2020.csv"

# Carregando o dataset
# Nota: Este dataset específico usa separador de vírgula padrão
df = pd.read_csv(url)

# Visualizando as primeiras linhas para o seu relatório
print("Dataset da Bram Offshore - Manutenção Preditiva")
display(df.head())

# DICA PARA O ITEM 2: Separação de X e y
# A coluna 'Machine failure' é o nosso alvo (target)
X = df[['Air temperature [K]', 'Process temperature [K]', 'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]']]
y = df['Machine failure']

Dataset da Bram Offshore - Manutenção Preditiva


,UDI,Product ID,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,TWF,HDF,PWF,OSF,RNF
0,1,M14860,M,298.1,308.6,1551,42.8,0,0,0,0,0,0,0
1,2,L47181,L,298.2,308.7,1408,46.3,3,0,0,0,0,0,0
2,3,L47182,L,298.1,308.5,1498,49.4,5,0,0,0,0,0,0
3,4,L47183,L,298.2,308.6,1433,39.5,7,0,0,0,0,0,0
4,5,L47184,L,298.2,308.7,1408,40.0,9,0,0,0,0,0,0


Separação de Treino e Teste (Holdout)
Para avaliar a capacidade de generalização do modelo e evitar o sobreajuste (overfitting), vamos dividir o dataset em dois conjuntos: um para treinar os algoritmos e outro exclusivo para testá-los. Utilizaremos a proporção de 80% para treino e 20% para teste. Como falhas em equipamentos costumam ser eventos raros, utilizaremos o parâmetro stratify=y para garantir que a mesma proporção de falhas seja mantida em ambos os conjuntos.

In [8]:
from sklearn.model_selection import train_test_split

# Executando o holdout (80% treino, 20% teste)
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42, # Garante que o resultado seja o mesmo toda vez que rodar
    stratify=y       # Mantém a proporção da classe alvo (Machine failure)
)

# Exibindo o resultado da separação
print(f"Total de amostras originais: {X.shape[0]}")
print(f"Tamanho do conjunto de Treino: {X_train.shape[0]} amostras")
print(f"Tamanho do conjunto de Teste: {X_test.shape[0]} amostras")

Total de amostras originais: 10000
Tamanho do conjunto de Treino: 8000 amostras
Tamanho do conjunto de Teste: 2000 amostras


### Transformação de Dados e Construção dos Pipelines

Com os dados de treino e teste separados, a próxima etapa é a modelagem. Conforme os requisitos do projeto, testaremos quatro algoritmos clássicos de classificação: **KNN, Árvore de Classificação, Naive Bayes e SVM**.

No entanto, antes de treinar os modelos, precisamos realizar a **Transformação dos Dados**.

**1. A Necessidade de Padronização (StandardScaler)**
Observando as características do nosso dataset (equipamentos offshore), notamos que as variáveis operam em grandezas físicas e escalas muito diferentes. Por exemplo, a *Rotational speed* atinge milhares de RPMs, enquanto o *Torque* ou a *Temperatura* operam em dezenas ou centenas. Algoritmos baseados no cálculo de distâncias (como o KNN e o SVM) são severamente prejudicados por essas diferenças de escala, pois a variável com maior magnitude dominará o cálculo. Para corrigir isso, aplicaremos a padronização utilizando o `StandardScaler`, que transforma as variáveis para que tenham média 0 e desvio padrão 1.

**2. A Escolha por Pipelines (Prevenção de Data Leakage)**
Para aplicar essa transformação de forma segura, adotaremos a boa prática de utilizar `Pipelines` da biblioteca Scikit-Learn.

O uso do Pipeline é fundamental para evitar o **vazamento de dados (data leakage)**. Se padronizássemos todo o dataset antes do holdout ou da validação cruzada, informações do conjunto de teste "vazariam" para o modelo durante o treinamento (através da média e variância globais). O Pipeline garante que o `StandardScaler` "aprenda" os parâmetros de escala **exclusivamente com os dados de treino** em cada iteração, aplicando essa mesma transformação aos dados de teste de forma cega, simulando perfeitamente um ambiente real de produção na Bram Offshore.

Abaixo, instanciamos os pipelines para os quatro modelos propostos.

In [9]:
# Importando as bibliotecas necessárias para os Pipelines e Modelos
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC

# Criando um dicionário de pipelines para facilitar o treinamento e comparação
pipelines = {
    "KNN": Pipeline([
        ("scaler", StandardScaler()),
        ("classifier", KNeighborsClassifier())
    ]),

    "Árvore de Decisão": Pipeline([
        ("scaler", StandardScaler()),
        ("classifier", DecisionTreeClassifier(random_state=42))
    ]),

    "Naive Bayes": Pipeline([
        ("scaler", StandardScaler()),
        ("classifier", GaussianNB())
    ]),

    "SVM": Pipeline([
        ("scaler", StandardScaler()),
        ("classifier", SVC(random_state=42))
    ])
}

print("Pipelines configurados com sucesso para os 4 algoritmos!")

Pipelines configurados com sucesso para os 4 algoritmos!


### Treinamento e Avaliação da Linha de Base (Baseline)

Com os pipelines construídos, vamos treinar os quatro modelos utilizando seus hiperparâmetros padrão. O objetivo desta etapa é estabelecer uma **linha de base de desempenho**.

Para a avaliação, utilizaremos as seguintes métricas:
* **Acurácia:** A taxa de acertos geral do modelo.
* **Relatório de Classificação (Precision, Recall, F1-Score):** Como estamos lidando com um problema de manutenção (onde prever uma falha é crítico), o *Recall* (capacidade de encontrar todas as falhas reais) e o *F1-Score* (média harmônica entre precisão e recall) são métricas essenciais.
* **Matriz de Confusão:** Para visualizar os Falsos Positivos e Falsos Negativos.

Através de um laço de repetição, treinaremos todos os pipelines, faremos as predições no conjunto de teste (`X_test`) e consolidaremos os resultados para comparação.

In [10]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import pandas as pd

# Lista para armazenar o resumo dos resultados
resultados = []

# Laço para iterar sobre o dicionário de pipelines
for nome, pipeline in pipelines.items():

    # 1. Treinamento (Fit) usando apenas os dados de treino
    pipeline.fit(X_train, y_train)

    # 2. Predição (Predict) usando os dados de teste (nunca vistos pelo modelo)
    y_pred = pipeline.predict(X_test)

    # 3. Cálculo das métricas
    acuracia = accuracy_score(y_test, y_pred)

    # Armazenando a acurácia para a tabela final
    resultados.append({
        'Modelo': nome,
        'Acurácia': round(acuracia, 4)
    })

    # 4. Exibição detalhada por modelo
    print(f"================ {nome} ================")
    print("Relatório de Classificação:")
    print(classification_report(y_test, y_pred))
    print("Matriz de Confusão:")
    print(confusion_matrix(y_test, y_pred))
    print("\n")

# Transformando os resultados em um DataFrame para facilitar a visualização e comparação
df_resultados = pd.DataFrame(resultados).sort_values(by='Acurácia', ascending=False)

print("=== Tabela Comparativa de Desempenho (Linha de Base) ===")
display(df_resultados)

================ KNN ================
Relatório de Classificação:
              precision    recall  f1-score   support

           0       0.98      1.00      0.99      1932
           1       0.85      0.34      0.48        68

    accuracy                           0.98      2000
   macro avg       0.91      0.67      0.74      2000
weighted avg       0.97      0.98      0.97      2000

Matriz de Confusão:
[[1928    4]
 [  45   23]]


================ Árvore de Decisão ================
Relatório de Classificação:
              precision    recall  f1-score   support

           0       0.99      0.99      0.99      1932
           1       0.69      0.65      0.67        68

    accuracy                           0.98      2000
   macro avg       0.84      0.82      0.83      2000
weighted avg       0.98      0.98      0.98      2000

Matriz de Confusão:
[[1912   20]
 [  24   44]]


================ Naive Bayes ================
Relatório de Classificação:
              precision    r

,Modelo,Acurácia
1,Árvore de Decisão,0.9780
0,KNN,0.9755
3,SVM,0.9720
2,Naive Bayes,0.9600


### Otimização de Hiperparâmetros com GridSearchCV

Com a linha de base estabelecida, selecionamos a **Árvore de Decisão** para a etapa de otimização de hiperparâmetros. Árvores de decisão são altamente interpretáveis, o que é um fator crítico em sistemas de suporte à decisão no setor offshore, permitindo que os engenheiros entendam quais regras levaram o modelo a prever uma falha.

Para encontrar a melhor configuração do modelo e evitar o *overfitting* (sobreajuste), utilizaremos o `GridSearchCV`. Esta técnica testa exaustivamente uma grade de hiperparâmetros combinada com a Validação Cruzada (*Cross-Validation*). A validação cruzada garante que o desempenho do modelo seja robusto e não dependa de uma única divisão de dados.

Os hiperparâmetros que iremos testar são:
* `criterion`: A métrica de qualidade da divisão dos nós (Gini ou Entropia).
* `max_depth`: A profundidade máxima da árvore (para evitar que ela decore os dados de treino).
* `min_samples_split`: O número mínimo de amostras necessárias para dividir um nó interno.

In [11]:
from sklearn.model_selection import GridSearchCV

# Definindo a grade de hiperparâmetros.
# O prefixo 'classifier__' é obrigatório pois estamos dentro de um Pipeline
param_grid = {
    'classifier__criterion': ['gini', 'entropy'],
    'classifier__max_depth': [None, 5, 10, 15, 20],
    'classifier__min_samples_split': [2, 5, 10, 20]
}

# Configurando o GridSearchCV com Cross-Validation de 5 folds (cv=5)
grid_search = GridSearchCV(
    estimator=pipelines["Árvore de Decisão"],
    param_grid=param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1 # Utiliza todos os núcleos do processador para acelerar a busca
)

print("Iniciando a busca com Cross-Validation. Aguarde...")
grid_search.fit(X_train, y_train)

# Resgatando o melhor modelo encontrado
melhor_modelo = grid_search.best_estimator_

print("\n=== Resultados da Otimização ===")
print(f"Melhores parâmetros encontrados:\n{grid_search.best_params_}")

# Testando o melhor modelo com os dados de teste (que estavam separados)
y_pred_otimizado = melhor_modelo.predict(X_test)
acuracia_otimizada = accuracy_score(y_test, y_pred_otimizado)

print(f"\nAcurácia do modelo otimizado nos dados de Teste: {acuracia_otimizada:.4f}")

Iniciando a busca com Cross-Validation. Aguarde...

=== Resultados da Otimização ===
Melhores parâmetros encontrados:
{'classifier__criterion': 'gini', 'classifier__max_depth': 10, 'classifier__min_samples_split': 20}

Acurácia do modelo otimizado nos dados de Teste: 0.9810


### Exportação do Modelo para Deploy

Com o modelo otimizado alcançando uma excelente performance (acurácia superior a 98%), a etapa final deste ciclo de machine learning é a sua exportação.

Para que este modelo possa ser integrado à aplicação Full Stack (back-end) da Bram Offshore, precisamos serializá-lo. Utilizaremos a biblioteca `joblib`, que é altamente eficiente para salvar objetos do Scikit-Learn, especialmente `Pipelines` que contêm passos de transformação de dados (como o nosso `StandardScaler`).

Ao exportar o pipeline completo, garantimos que os dados recebidos na aplicação web no futuro passarão exatamente pela mesma padronização antes da predição, evitando erros de escala e garantindo a confiabilidade do sistema de manutenção preditiva.

In [12]:
import joblib

# Definindo o nome do arquivo do modelo
nome_arquivo = 'modelo_manutencao_bram.pkl'

# Salvando o pipeline completo (StandardScaler + Árvore de Decisão otimizada)
joblib.dump(melhor_modelo, nome_arquivo)

print(f"✅ Modelo exportado com sucesso para o arquivo: '{nome_arquivo}'")
print("Para baixar: Clique no ícone de pasta (Arquivos) no menu lateral esquerdo do Colab,")
print("clique nos três pontinhos ao lado do arquivo .pkl e selecione 'Fazer o download'.")

✅ Modelo exportado com sucesso para o arquivo: 'modelo_manutencao_bram.pkl'
Para baixar: Clique no ícone de pasta (Arquivos) no menu lateral esquerdo do Colab,
clique nos três pontinhos ao lado do arquivo .pkl e selecione 'Fazer o download'.


## Conclusão e Análise de Resultados

Este projeto teve como objetivo desenvolver um modelo de *Machine Learning* capaz de prever falhas em equipamentos industriais (simulando ativos do mercado offshore, como propulsores e motores), traduzindo dados brutos de sensores em inteligência de suporte à decisão.

**Principais Achados e Desempenho:**
Após avaliar uma linha de base com algoritmos clássicos (KNN, Naive Bayes, SVM e Árvore de Decisão), optamos por otimizar a Árvore de Decisão devido à sua interpretabilidade — uma característica vital para a engenharia de manutenção. Através da técnica de *Cross-Validation* com `GridSearchCV`, o modelo atingiu uma acurácia excepcional de **98,10%** no conjunto de teste.

Os hiperparâmetros otimizados (`criterion='gini'`, `max_depth=10`, `min_samples_split=20`) revelam que o algoritmo conseguiu mapear regras complexas sem sofrer de *overfitting* (sobreajuste). A limitação da profundidade da árvore para 10 níveis impediu que o modelo "decorasse" ruídos no conjunto de treinamento, garantindo sua capacidade de generalização para dados inéditos.

**Pontos de Atenção (Análise Crítica):**
Apesar da alta acurácia, em contextos de manutenção preditiva offshore, o custo das classificações erradas é assimétrico. Um **Falso Negativo** (o modelo prever que o equipamento está normal quando, na verdade, está prestes a falhar) pode resultar em tempo de inatividade da embarcação (*downtime*) e altos custos operacionais. Portanto, em iterações futuras deste sistema, seria recomendável ajustar o limiar de decisão (*threshold*) para priorizar a métrica de *Recall* da classe de falha, aceitando um leve aumento nos Falsos Positivos (inspeções preventivas desnecessárias) em troca de não deixar passar nenhuma falha real.

**Fechamento e Próximos Passos:**
O fluxo de engenharia de software inteligente foi cumprido com sucesso: desde a carga e separação segura dos dados (holdout), mitigação de *data leakage* via Pipelines com `StandardScaler`, até a modelagem e otimização. O modelo resultante foi exportado serializado (`.pkl`) e agora está pronto para ser embarcado no *back-end* da nossa aplicação Full Stack, onde receberá dados de telemetria em tempo real e fornecerá diagnósticos instantâneos para as equipes de operação.

**Reflexão sobre Desenvolvimento de Software Seguro:**
No contexto de sistemas industriais marítimos (como na Bram Offshore), os dados de telemetria podem ocasionalmente estar atrelados a identificadores de tripulantes (ex: matrícula do operador da máquina) ou localizações estratégicas das embarcações. Para aplicar as boas práticas de Desenvolvimento de Software Seguro e conformidade com a LGPD, o dataset passaria por um processo de anonimização antes de alimentar o modelo de Machine Learning. Isso envolveria a exclusão de colunas de identificação pessoal ou o uso de técnicas de hashing (pseudo-anonimização) para o ID dos tripulantes, garantindo que o algoritmo aprenda apenas os padrões mecânicos de falha, sem expor dados sensíveis da equipe ou revelar a posição exata de ativos estratégicos. Além disso, a aplicação Full Stack deve sanitizar os inputs numéricos para evitar ataques de injeção (XSS/SQLi) vindos do front-end.